### GENAI APP USING LANGCHAIN AND WEBBASED LOADER

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
#os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN__TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
# Data Ingestion

from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.geeksforgeeks.org/artificial-intelligence/large-language-model-llm/")
docs = loader.load()
print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[Document(metadata={'source': 'https://www.geeksforgeeks.org/artificial-intelligence/large-language-model-llm/', 'title': 'Introduction to Large Language Model (LLM) - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en'}, page_content='Introduction to Large Language Model (LLM) - GeeksforGeeksCoursesTutorialsPracticeJobsArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsEarn Google BadgeIntroduction to AIWhat is Artificial Intelligence (AI)Types of Artificial Intelligence (AI)Types of AI Based on FunctionalitiesAgents in AIArtificial intelligence vs Machine Learning vs Deep LearningProblem Solving in Artificial IntelligenceA

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [4]:
documents

[Document(metadata={'source': 'https://www.geeksforgeeks.org/artificial-intelligence/large-language-model-llm/', 'title': 'Introduction to Large Language Model (LLM) - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en'}, page_content='Introduction to Large Language Model (LLM) - GeeksforGeeksCoursesTutorialsPracticeJobsArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsEarn Google BadgeIntroduction to AIWhat is Artificial Intelligence (AI)Types of Artificial Intelligence (AI)Types of AI Based on FunctionalitiesAgents in AIArtificial intelligence vs Machine Learning vs Deep LearningProblem Solving in Artificial IntelligenceA

In [5]:
# Embeddings

from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the embedding model (downloads automatically on first run)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

/home/aakash/anaconda3/envs/ds/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5063.63it/s]


In [6]:
# Vector Stores
from langchain_community.vectorstores import FAISS
vectorstoredb = FAISS.from_documents(documents, embedding=embeddings)

In [7]:
vectorstoredb

In [8]:
# query from a vector store db
query = " Self-Attention"
result = vectorstoredb.similarity_search(query)

In [9]:
location = result[0].page_content.find("Self-Attention")
print(result[0].page_content[location-100:location+100])

and feed-forward networks.A simplified flow is: Tokens → Embeddings → Transformer Blocks → Output4. Self-AttentionSelf-attention helps the model understand relationships between tokens by determining 


In [10]:
from langchain_community.llms import Ollama
llm = Ollama(model="gemma2:9b", base_url="http://localhost:11434")

/tmp/ipykernel_9480/1251122211.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma2:9b", base_url="http://localhost:11434")


In [11]:
# Retrieval Chain,Document Chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
prompt = ChatPromptTemplate.from_template(""" 
Answer the following question based on the provided context:
<context>{context}</context>
""")

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the following question based on the provided context:\n<context>{context}</context>\n'), additional_kwargs={})])
| Ollama(model='gemma2:9b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [12]:
from langchain_core.documents import Document
document_chain.invoke({"input":"By using massive datasets and billions of parameters, LLMs have transformed the way humans interact with technology.",
                      "context":[Document(page_content="By using massive datasets and billions of parameters, LLMs have transformed the way humans interact with technology.")]})

'What has transformed the way humans interact with technology? \n\n**Answer:** LLMs (Large Language Models) have transformed the way humans interact with technology. \n'

In [13]:
# Retriever-->An interface to fetch info from vectorstore db
retriever = vectorstoredb.as_retriever()

In [14]:
from langchain.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [15]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x766d39793e00>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template=' \nAnswer the following question based on the provided context:\n<context>{context}</context>\n'), additional_kwargs={})])
            | Ollama(mode

In [16]:
# Get the response from the LLM
response = retrieval_chain.invoke({"input": "What is Self-Attention?"})
response['answer']

'What are some popular LLM families? \n\n\nThe provided text lists these popular LLM families:\n\n* **GPT** (OpenAI)\n* **Llama** (Meta)\n* **Claude** (Anthropic)\n* **Gemini** (Google DeepMind)\n* **Mistral** \n* **Qwen** (Alibaba) \n'

In [ ]:
response